In [3]:
import pandas as pd
import duckdb

# 投料数据：电商秒杀节注册与首单流
user_orders = pd.DataFrame({
    'order_id': ['o_01', 'o_02', 'o_03', 'o_04'],
    'user_id': [9001, 9002, 9003, 9004],
    'signup_time': ['2026-06-21 00:00:00', '2026-06-21 06:00:00', '2026-06-21 12:00:00', '2026-06-21 23:00:00'],
    'order_time': ['2026-06-21 01:15:00', '2026-06-21 10:30:00', '2026-06-21 12:05:00', '2026-06-22 03:00:00']
})

### 🎯 2. 核心刚性需求

1. **锁定风控安全线**：利用 `INTERVAL` 算子，计算出每个用户注册后的**黑产高危观察截止时间**（即 `signup_time` 往后平移 **2 小时**）。
    
2. **大闸拦截**：如果下单时间（`order_time`）严格落在了注册后的 2 小时观察期内，将其判定为高危单。
    
3. **提取特征**：利用 `EXTRACT(EPOCH FROM ...)` 或者是 Pandas 的平替算子，算出这批高危单到底是在注册后**多少秒内**发起的，命名为 `fraud_window_seconds`。
    
4. **最终输出**：输出被逮住的 `user_id` 和 `fraud_window_seconds`。

In [7]:
# =====================================================================
# ⚔️  轨道一：PostgreSQL 
# =====================================================================
sql_query = """
WITH format_conversion_stage AS (
SELECT  order_id,
        user_id,
        signup_time::TIMESTAMP AS signup_time,
        signup_time :: TIMESTAMP + INTERVAL '2 hours' AS signup_time_2hr,
        order_time :: TIMESTAMP AS order_time
FROM user_orders
),
filter_stage AS (
SELECT  user_id,
        order_id,
        signup_time,
        signup_time_2hr,
        order_time
FROM format_conversion_stage
WHERE order_time < signup_time_2hr
)
SELECT  user_id,
        EXTRACT(EPOCH FROM (order_time - signup_time)) :: INTEGER AS fraud_window_seconds
FROM filter_stage
"""
df_sql = duckdb.query(sql_query).df()
print(df_sql)

   user_id  fraud_window_seconds
0     9001                  4500
1     9003                   300


In [ ]:
# =====================================================================
# ⚔️  轨道二：PANDAS 
# =====================================================================
df_user_orders = (
    user_orders
    .assign(
        signup_time = pd.to_datetime(user_orders['signup_time']),
        order_time = pd.to_datetime(user_orders['order_time']),
        signup_time_2hr = lambda x:pd.to_datetime(x['signup_time']) + pd.Timedelta(hours=2)
    )
    .query(
        "order_time < signup_time_2hr"
    )
    .assign(
        fraud_window_seconds = lambda x:(x['order_time'] - x['signup_time'])
                                        .dt.total_seconds()
                                        .astype(int)
    )
    [['user_id','fraud_window_seconds']]
    .sort_values(by='user_id')
    .reset_index(drop=True)
)
print(df_user_orders)

   user_id  fraud_window_seconds
0     9001                  4500
2     9003                   300


In [12]:
pd.testing.assert_frame_equal(
    df_sql.reset_index(drop=True),
    df_user_orders.reset_index(drop=True),
    check_dtype=False
)
print("🏆【天衣无缝！】第二题风控对账全绿通过！SQL 和 Pandas 链式匿名函数流完美对齐！")

🏆【天衣无缝！】第二题风控对账全绿通过！SQL 和 Pandas 链式匿名函数流完美对齐！
